In [ ]:
!pip install pyspark

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving city_temperature(1).csv to city_temperature(1).csv
Saving country-list(1).csv to country-list(1).csv


In [ ]:
import os
os.listdir('.')

['.config', 'country-list(1).csv', 'city_temperature(1).csv', 'sample_data']

In [ ]:
import csv
from pyspark import SparkContext
import os

# Initialize Spark Context
sc = SparkContext.getOrCreate()
# Helper Function for Data Cleaning
def parse_temp_csv(line):
    # Use csv module to handle potential quotes and commas within fields
    reader = csv.reader([line])
    columns = next(reader)
    # Filter out header or malformed lines
    if not columns or columns[0] == "Region":
        return None
    try:
        # Columns: Region(0), Country(1), State(2), City(3), Month(4), Day(5), Year(6), AvgTemperature(7)
        region = columns[0]
        country = columns[1]
        city = columns[3]
        year = columns[6]
        temp = float(columns[7])

        # Most temperature datasets use -99 or -99.9 for missing data
        if temp <= -90:
            return None
        return (region, country, city, year, temp)
    except:
        return None
# Load the temperature dataset
temp_raw = sc.textFile("city_temperature(1).csv")
# Apply cleaning and filter out None values
temp_data = temp_raw.map(parse_temp_csv).filter(lambda x: x is not None).cache()

# Part A: Min/Max Temperature per Region
# Map to (Region, (Temp, Temp)) where the tuple is (min_candidate, max_candidate)
region_min_max = temp_data.map(lambda x: (x[0], (x[4], x[4]))) \
                          .reduceByKey(lambda a, b: (min(a[0], b[0]), max(a[1], b[1])))
print(" Part A: Min and Max Temp per Region")
for region, (min_t, max_t) in region_min_max.collect():
    print(f"{region}: Min={min_t}, Max={max_t}")
# Part B: Avg Temp by Year for "Europe"
# Filter for Europe, map to (Year, (Temp, 1)) to calculate sum and count
europe_avg = temp_data.filter(lambda x: x[0] == "Europe") \
                     .map(lambda x: (x[3], (x[4], 1))) \
                     .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
                     .mapValues(lambda x: x[0] / x[1]) \
                     .sortByKey()
print("\n Part B: Avg Temp in Europe by Year")
print(europe_avg.collect())

# Part C: Top 10 Cities in "United States"
# Filter for US, map to (City, (Temp, 1)), then calculate average
us_top_cities = temp_data.filter(lambda x: x[1] == "US") \
                         .map(lambda x: (x[2], (x[4], 1))) \
                         .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
                         .mapValues(lambda x: x[0] / x[1]) \
                         .takeOrdered(10, key=lambda x: -x[1])
print("\n Part C: Top 10 Warmest US Cities")
for city, avg_temp in us_top_cities:
    print(f"{city}: {avg_temp:.2f}")

# Helper: Function to save results
def save_rdd(rdd, folder_name):
    import os
    if os.path.exists(folder_name):
        os.system(f"rm -rf {folder_name}")
    rdd.saveAsTextFile(folder_name)
# Part D: Capital City Avg Temp by Year
# Parse the country-list file
def parse_country_list(line):
    reader = csv.reader([line])
    cols = next(reader)
    if not cols or cols[0] == "country": return None
    # Returns (CapitalCity, Country) -> e.g., ("London", "United Kingdom")
    return (cols[1].strip(), cols[0].strip())
country_rdd = sc.textFile("country-list(1).csv") \
                .map(parse_country_list) \
                .filter(lambda x: x is not None)
# Map temp_data to (City, (Year, Temp)) to prepare for the join
# index 2 is City, index 3 is Year, index 4 is Temp
temp_to_join = temp_data.map(lambda x: (x[2].strip(), (x[3].strip(), x[4])))
# Join the datasets on City Name
# Result format: (City, ((Year, Temp), Country))
joined_data = temp_to_join.join(country_rdd)
# Group by (Country, Year) and calculate Average
# We use x[1][1] for Country and x[1][0][0] for Year
results_d = joined_data.map(lambda x: ((x[1][1], x[1][0][0]), (x[1][0][1], 1))) \
                       .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
                       .mapValues(lambda x: x[0] / x[1]) \
                       .sortByKey() \
                       .map(lambda x: f"{x[0][0]}\t{x[0][1]}\t{x[1]:.2f}")
# Save the output
save_rdd(results_d, "q2_partD_results")
print("\n Part D: Sorted Capital Temp Results")
for row in results_d.take(10):
    print(row)

 Part A: Min and Max Temp per Region
Asia: Min=-37.2, Max=103.7
Africa: Min=33.3, Max=102.8
Australia/South Pacific: Min=30.7, Max=96.8
South/Central America & Carribean: Min=32.8, Max=97.4
Europe: Min=-20.4, Max=102.5
Middle East: Min=3.2, Max=110.0
North America: Min=-50.0, Max=107.7

 Part B: Avg Temp in Europe by Year
[('1995', 51.539997336174785), ('1996', 50.13843532870105), ('1997', 51.44022328132136), ('1998', 51.86676074827715), ('1999', 52.25095508181753), ('2000', 53.021256743890696), ('2001', 51.981835600689614), ('2002', 52.51161844484637), ('2003', 52.207324736532016), ('2004', 51.79084399655254), ('2005', 51.77056006385521), ('2006', 52.119745179532615), ('2007', 52.90528906141684), ('2008', 52.48594328820629), ('2009', 52.36746484755676), ('2010', 50.53998198697511), ('2011', 52.31573828470382), ('2012', 51.44140801869579), ('2013', 51.21466558032722), ('2014', 52.76514442711818), ('2015', 52.61556512698664), ('2016', 52.009337911472635), ('2017', 52.05164043942989), ('

In [ ]:
import csv
import os
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

def save_rdd(rdd, folder_name):
    if os.path.exists(folder_name):
        os.system(f"rm -rf {folder_name}")
    rdd.saveAsTextFile(folder_name)

def parse_temp_csv(line):
    reader = csv.reader([line])
    columns = next(reader)
    if not columns or columns[0] == "Region": return None
    try:
        # Region(0), Country(1), City(3), Year(6), Temp(7)
        return (columns[0].strip(), columns[1].strip(), columns[3].strip(), columns[6].strip(), float(columns[7]))
    except: return None

temp_data = sc.textFile("city_temperature(1).csv") \
              .map(parse_temp_csv) \
              .filter(lambda x: x is not None and x[4] > -90).cache()

# Part A: Min/Max per Region
res_a = temp_data.map(lambda x: (x[0], (x[4], x[4]))) \
                 .reduceByKey(lambda a, b: (min(a[0], b[0]), max(a[1], b[1]))) \
                 .map(lambda x: f"{x[0]}\t{x[1][0]}\t{x[1][1]}")
save_rdd(res_a, "q2_partA_results")

# Part B: Europe Avg
res_b = temp_data.filter(lambda x: x[0] == "Europe") \
                 .map(lambda x: (x[3], (x[4], 1))) \
                 .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
                 .mapValues(lambda x: x[0] / x[1]) \
                 .sortByKey() \
                 .map(lambda x: f"{x[0]}\t{x[1]}")
save_rdd(res_b, "q2_partB_results")

# Part C: Top 10 US Cities
# Note: Using "US" as determined earlier
top_10_us_list = temp_data.filter(lambda x: x[1] == "US") \
                          .map(lambda x: (x[2], (x[4], 1))) \
                          .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
                          .mapValues(lambda x: x[0] / x[1]) \
                          .takeOrdered(10, key=lambda x: -x[1])
res_c = sc.parallelize(top_10_us_list).map(lambda x: f"{x[0]}\t{x[1]:.2f}")
save_rdd(res_c, "q2_partC_results")

# Part D: Capitals
def parse_country_list(line):
    reader = csv.reader([line])
    cols = next(reader)
    if not cols or cols[0] == "country": return None
    return (cols[1].strip(), cols[0].strip())

country_rdd = sc.textFile("country-list(1).csv").map(parse_country_list).filter(lambda x: x is not None)
temp_to_join = temp_data.map(lambda x: (x[2], (x[3], x[4])))
joined = temp_to_join.join(country_rdd)

res_d = joined.map(lambda x: ((x[1][1], x[1][0][0]), (x[1][0][1], 1))) \
              .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
              .mapValues(lambda x: x[0] / x[1]) \
              .map(lambda x: f"{x[0][0]}\t{x[0][1]}\t{x[1]:.2f}")
save_rdd(res_d, "q2_partD_results")

print("Question 2 outputs saved successfully.")

Question 2 outputs saved successfully.


In [ ]:
!zip -r Q2_Results.zip q2_partA_results q2_partB_results q2_partC_results q2_partD_results

  adding: q2_partA_results/ (stored 0%)
  adding: q2_partA_results/.part-00001.crc (stored 0%)
  adding: q2_partA_results/.part-00004.crc (stored 0%)
  adding: q2_partA_results/.part-00000.crc (stored 0%)
  adding: q2_partA_results/part-00002 (stored 0%)
  adding: q2_partA_results/.part-00003.crc (stored 0%)
  adding: q2_partA_results/part-00000 (stored 0%)
  adding: q2_partA_results/part-00001 (stored 0%)
  adding: q2_partA_results/part-00003 (stored 0%)
  adding: q2_partA_results/_SUCCESS (stored 0%)
  adding: q2_partA_results/._SUCCESS.crc (stored 0%)
  adding: q2_partA_results/part-00004 (stored 0%)
  adding: q2_partA_results/.part-00002.crc (stored 0%)
  adding: q2_partB_results/ (stored 0%)
  adding: q2_partB_results/.part-00001.crc (stored 0%)
  adding: q2_partB_results/.part-00004.crc (stored 0%)
  adding: q2_partB_results/.part-00000.crc (stored 0%)
  adding: q2_partB_results/part-00002 (deflated 39%)
  adding: q2_partB_results/.part-00003.crc (stored 0%)
  adding: q2_partB_re